In [0]:
%sql
-- ============================================================
-- PIPELINE: Bronze → Silver
-- ============================================================

INSERT OVERWRITE analisis_tmdb.silver.genres

WITH cleaned_data AS (

    -- 1 Limpieza básica + conversión de tipos
    SELECT
        TRY_CAST(genre_id AS INT) AS genre_id,
        TRIM(genre_name) AS genre_name,
        ingestion_timestamp
    FROM analisis_tmdb.bronze.genres_bronze
    WHERE genre_id IS NOT NULL
      AND genre_name IS NOT NULL
      AND genre_name <> ''

),

deduplicated AS (

    -- 2 Deduplicación
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY genre_id ORDER BY ingestion_timestamp DESC) AS rn
        FROM cleaned_data
    )
    WHERE rn = 1
),

final_dataset AS (

    -- 3 Dataset final
    SELECT

        genre_id,
        genre_name,
        ingestion_timestamp

    FROM deduplicated

)

-- 4 Insert final a silver
SELECT *
FROM final_dataset;

In [0]:
-- ============================================================
-- VERIFICACIÓN: Comparar Bronze vs Silver
-- ============================================================

SELECT 
    'Bronze (raw)' as capa,
    COUNT(*) as registros
FROM analisis_tmdb.bronze.genres_bronze

UNION ALL

SELECT 
    'Silver' as capa,
    COUNT(*) as registros
FROM analisis_tmdb.silver.genres;